In [ ]:
import numpy as np
import math

d = 50
n = 7

#upper_bound = ((n*d*(1-((d-1)/d)**2))/(1-n*d*((d-1)/d)**2))**(1/d)   #WRONG
#upper_bound = (d*n*d*(n*d+1)/2)**(1/d)  #WRONG
# upper_bound = math.comb(n*d, d)**(1/d)
# upper_bound = (n*math.comb(d, d))**(1/d)
lower_bound = (0.5+1/8*(3/n)**d+1/2*(n/3)**d)**(1/d) #PROBABILISTIC BOUND
# upper_bound = (d*(d+1))**(1/d)

# print(f"Upper bound: {upper_bound}")
print(f"Lower bound: {lower_bound}")


Lower bound: 2.301209643817838
0.9994150254340944


In [6]:
x = (2/0.0625)**(1/3)
d = 3
y = (2/7*math.ceil(3.25**d))**(1/(d-1))
print(x)
print(y)

# for i in range(112):
#     A = math.floor(i*2)%7
#     B = math.floor(i*2/x)%7
#     C = math.floor(i*2/x/x)%7
#     D = math.floor(i*2/x/x/x)%7
#     print(i, A, B, C, D)

# for i in range(112):
#     A = math.floor(i*2)%7
#     B = math.floor(i*2/x)%7
#     C = math.floor(i*2/x/x)%7
#     D = math.floor(i*2/x/x/x)%7
#     print(i, A, B, C, D)

3.1748021039363987
3.1622776601683795


In [7]:
from itertools import product
import math
import numpy as np
from tqdm import tqdm

def generate_shifts(d, shift_size):
    # Implementation for generating shifts
    # Generate all combinations of sets of integers of dimension d and largest integer shift_size-1, first integer can be larger than second etc
    shifts = np.array(list(product(range(shift_size), repeat=d)))
    return shifts

def count_adjacencies(point, points, p):
    count = 0
    #If for any dimension d , the absolute difference is larger than 1, not adjacent
    for other in points:
        is_adjacent = True
        for dim in range(len(point)):
            diff = abs(point[dim] - other[dim])
            if diff > 1 and diff < (p-1):
                is_adjacent = False
                break
        if is_adjacent:
            count += 1    
    return count

def return_adjacencies(points, p):
    adjacency_counts = []
    for point in points:
        count = count_adjacencies(point, points, p)
        adjacency_counts.append(count-1)  # Subtract 1 to not count itself
    return adjacency_counts

def create_independent_points(points, p):
    adjacency_counts = return_adjacencies(points, p)
    while (max(adjacency_counts) > 0):
        print(f"There are in total {np.count_nonzero(adjacency_counts)} nonzero adjacencies")
        max_index = adjacency_counts.index(max(adjacency_counts))
        points = np.delete(points, max_index, axis=0)
        adjacency_counts = return_adjacencies(points, p)
    return points

def generate_all_points(n, d):
    total_points = n ** d
    points = np.zeros((total_points, d), dtype=int)
    
    for i in range(total_points):
        for dim in range(d):
            points[i, dim] = (i // (n ** dim)) % n
    return points


def find_available_points(ind_set, other_set, d, p):
    available_points = []
    for point in other_set:
        is_available = True
        for ind_point in ind_set:
            # Check adjacency
            is_adjacent = True
            for dim in range(d):
                diff = abs(point[dim] - ind_point[dim])
                if diff > 1 and diff < (p-1):
                    is_adjacent = False
                    break
            if is_adjacent:
                is_available = False
                break
        if is_available:
            available_points.append(point)
    
    return np.array(available_points)

def extend_independent_set(ind_set, n, d):
    all_points = generate_all_points(n, d)
    other_set = np.array([point for point in all_points if point.tolist() not in ind_set.tolist()])
    available_points = find_available_points(ind_set, other_set, d, n)
    print(f"Found {len(available_points)} available points to extend the independent set.")
    while len(available_points) > 0:
        adjacency_counts = return_adjacencies(available_points, n)
        min_index = adjacency_counts.index(min(adjacency_counts))
        ind_set = np.vstack([ind_set, available_points[min_index]])
        available_points = np.delete(available_points, min_index, axis=0)
        available_points = find_available_points(ind_set, available_points, d, n)

    return ind_set

def shift_set(p, large_set):
    n = len(large_set)
    d = len(large_set[0])
    shift_size = math.ceil(2*(n-1)/p)
    print(f"Start generating shifts, shiftsize is {shift_size}")
    # shifts = generate_shifts(d, shift_size)
    # shifts = np.array([[40,123,40,123,40]])
    # print(f"Generated {len(shifts)} shifts.")
    # for shift in shifts:
    i = 0
    while i < 100:
        shift = np.random.randint(0, shift_size, size=d)
        # # if shift.tolist() == [40,14,40,14,40]:
        # #     print(((shifted_set - np.floor(((large_set + shift) % n) / shift_size))%p).tolist())
        shifted_set = np.floor(((large_set + shift) % n) / (shift_size/2))
        independent_points = create_independent_points(shifted_set, p)
        print(f"Created independent set of size {len(independent_points)} with q_array {q_array}")
        extended_set = extend_independent_set(independent_points, p, d)
        print(f"Extended to independent set of size {len(extended_set)} using the shift {shift}")
        i += 1

p = 7
q = 39
d = 6
n = 1146
print(f"Done importing")
q_array = np.array([q**i for i in range(d)])
large_set = (q_array * np.arange(n)[:, np.newaxis])%n
print(f"Generated large set of size {len(large_set)}")
shift_set(p, large_set)

Done importing
Generated large set of size 1146
Start generating shifts, shiftsize is 328


KeyboardInterrupt: 

In [6]:
import numpy as np
from tqdm import tqdm

from itertools import product
import math

def generate_shifts(d, shift_size):
    # Implementation for generating shifts
    # Generate all combinations of sets of integers of dimension d and largest integer shift_size-1, first integer can be larger than second etc
    shifts = np.array(list(product(range(shift_size), repeat=d)))
    return shifts

def check_adjacent(point1, point2, p, k):
    for dim in range(len(point1)):
        diff = abs(point1[dim] - point2[dim])
        if diff > (k-1) and diff < (p-(k-1)):
            return False
    return True

def count_adjacencies(point, points, p, k):
    count = 0
    #If for any dimension d , the absolute difference is larger than 1, not adjacent
    for other in points:
        if check_adjacent(point, other, p, k)==True:
            count+=1   
    return count

def return_adjacencies(points, p, k):
    adjacency_counts = []
    for point in points:
        count = count_adjacencies(point, points, p, k)
        adjacency_counts.append(count-1)  # Subtract 1 to not count itself
    return adjacency_counts

def create_independent_points(points, p, k):
    adjacency_counts = return_adjacencies(points, p, k)
    while (max(adjacency_counts) > 0):
        # max_index = adjacency_counts.index(max(adjacency_counts))
        # Set max_index random among the indices that are positive in adjacency_counts
        nonzero_indices = [i for i, count in enumerate(adjacency_counts) if count > 0]
        # max_index = np.random.choice(nonzero_indices)
        max_index = nonzero_indices[0]
        points = np.delete(points, nonzero_indices, axis=0)
        adjacency_counts = return_adjacencies(points, p, k)
    return points

def generate_all_points(n, d):
    total_points = n ** d
    points = np.zeros((total_points, d), dtype=int)
    
    for i in range(total_points):
        for dim in range(d):
            points[i, dim] = (i // (n ** dim)) % n
    return points


def find_available_points(ind_set, other_set, d, p, k):
    available_points = []
    for point in other_set:
        is_available = True
        for ind_point in ind_set:
            # Check adjacency
            if check_adjacent(ind_point, point, p, k):
                is_available = False
                break
        if is_available:
            available_points.append(point)
    
    return np.array(available_points)

def extend_independent_set(ind_set, n, d, k):
    all_points = generate_all_points(n, d)
    other_set = np.array([point for point in all_points if point.tolist() not in ind_set.tolist()])
    available_points = find_available_points(ind_set, other_set, d, n, k)
    print(f"Found {len(available_points)} available points to extend the independent set.")
    while len(available_points) > 0:
        adjacency_counts = return_adjacencies(available_points, n, k)
        min_index = adjacency_counts.index(min(adjacency_counts))
        ind_set = np.vstack([ind_set, available_points[min_index]])
        available_points = np.delete(available_points, min_index, axis=0)
        available_points = find_available_points(ind_set, available_points, d, n, k)

    return ind_set

def shift_set(p, large_set):
    n = len(large_set)
    d = len(large_set[0])
    shift_size = math.ceil(2*(n-1)/p)
    print(f"Start generating shifts, shiftsize is {shift_size}")
    # shifts = generate_shifts(d, shift_size)
    # shifts = np.array([[40,123,40,123,40]])
    # print(f"Generated {len(shifts)} shifts.")
    # for shift in shifts:
    i = 0
    while i < 1:
        # q_array = np.ones(5)
        # for j in range(4):
        #     q_array[j+1] = np.random.randint(q_array[j]+1, n-1-(3-j))
        # large_set = (q_array * np.arange(n)[:, np.newaxis])%n
        # shift = np.random.randint(0, shift_size, size=d)
        # # if shift.tolist() == [40,14,40,14,40]:
        # #     print(((shifted_set - np.floor(((large_set + shift) % n) / shift_size))%p).tolist())
        shift = np.array([40,124,40,123,40])
        shifted_set = np.floor(((large_set + shift) % n) / (shift_size/2))
        independent_points = create_independent_points(shifted_set, p, 2)
        print(f"Created independent set of size {len(independent_points)} with q_array {q_array}")
        extended_set = extend_independent_set(independent_points, p, d, 2)
        print(f"Extended to independent set of size {len(extended_set)} using the shift {shift}")
        i += 1
    return independent_points


def maximum_independent_set_exact(points, check_adjacency):
    """
    Exact maximum independent set via branch-and-bound with bitsets.

    Parameters
    ----------
    points : array-like of shape (N, d)
        Candidate vertices.
    check_adjacency : callable
        Function check_adjacency(p1, p2) -> bool where True means
        p1 and p2 are adjacent in the conflict graph (so they cannot both
        be in the independent set).

    Returns
    -------
    best_indices : list[int]
        Indices of vertices in a maximum independent set.
    best_points : np.ndarray
        The corresponding points.
    """
    points = np.asarray(points)
    n = len(points)

    if n == 0:
        return [], np.empty((0, 0), dtype=int)

    # Build adjacency bitmasks of the conflict graph.
    adj = [0] * n
    num_edges = 0
    for i in range(n):
        for j in range(i + 1, n):
            if check_adjacency(points[i], points[j]):
                adj[i] |= 1 << j
                adj[j] |= 1 << i
                num_edges += 1

    print(f"Graph has {n} nodes and {num_edges} edges")

    full_mask = (1 << n) - 1
    best_set = 0
    best_size = 0

    def popcount(x):
        return x.bit_count()

    def choose_vertex(candidates):
        # Branch first on a high-degree vertex for stronger pruning.
        x = candidates
        best_v = -1
        best_deg = -1
        while x:
            lsb = x & -x
            v = lsb.bit_length() - 1
            deg = popcount(adj[v] & candidates)
            if deg > best_deg:
                best_deg = deg
                best_v = v
            x ^= lsb
        return best_v

    def dfs(candidates, current_set, current_size):
        nonlocal best_set, best_size

        # Upper bound: even if we take all candidates, cannot beat current best.
        if current_size + popcount(candidates) <= best_size:
            return

        if candidates == 0:
            if current_size > best_size:
                best_size = current_size
                best_set = current_set
            return

        v = choose_vertex(candidates)
        bit_v = 1 << v

        # Include v: remove v and all its neighbors from candidates.
        dfs(candidates & ~adj[v] & ~bit_v, current_set | bit_v, current_size + 1)

        # Exclude v.
        dfs(candidates & ~bit_v, current_set, current_size)

    dfs(full_mask, 0, 0)

    best_indices = [i for i in range(n) if (best_set >> i) & 1]
    best_points = points[best_indices]
    return best_indices, best_points


# Example adapter for your existing signature check_adjacent(point1, point2, p, k)
# Replace p and k with your current values.
def make_check_adjacency(p, k):
    return lambda a, b: check_adjacent(a, b, p, k)

def experiment(p, large_set):
    n = len(large_set)
    d = len(large_set[0])
    k = 2
    shift_size = math.ceil(2*(n-1)/p)
    print(f"Start generating shifts, shiftsize is {shift_size}")
    # shifts = generate_shifts(d, shift_size)
    # shifts = np.array([[40,123,40,123,40]])
    # print(f"Generated {len(shifts)} shifts.")
    # for shift in shifts:
    all_points = generate_all_points(p, d)
    max_l = 0
    while max_l < 367:
        # q_array = np.ones(5)
        # for j in range(4):
        #     q_array[j+1] = np.random.randint(q_array[j]+1, n-1-(3-j))
        # large_set = (q_array * np.arange(n)[:, np.newaxis])%n
        # shift = np.random.randint(0, shift_size, size=d)
        # # if shift.tolist() == [40,14,40,14,40]:
        # #     print(((shifted_set - np.floor(((large_set + shift) % n) / shift_size))%p).tolist())
        shift = np.array([40,124,40,123,40])
        shifted_set = np.floor(((large_set + shift) % n) / (shift_size/2))
        independent_points = create_independent_points(shifted_set, p, 2)
        print(f"Created independent set of size {len(independent_points)} with q_array {q_array}")
        while len(independent_points) > 327:
            independent_points = np.delete(independent_points, np.random.randint(0, len(independent_points)), axis=0)
        other_set = np.array([point for point in all_points if point.tolist() not in independent_points.tolist()])
        available_points = find_available_points(independent_points, other_set, d, p, k)
        print(f"Found {len(available_points)} available points to extend the independent set.")
        check_adjacency = make_check_adjacency(p, k)
        best_idx, best_independent_set = maximum_independent_set_exact(available_points, check_adjacency)
        length = len(best_idx)+len(independent_points)
        print("Maximum independent set size:", length)
        if length > max_l:
            max_l = length
            max_set = np.vstack([best_independent_set, independent_points])

    return max_l, max_set

p = 7
q = 39
d = 5
n = 382
print(f"Done importing")
q_array = np.array([q**i for i in range(d)])
large_set = (q_array * np.arange(n)[:, np.newaxis])%n
max_l, max_set = experiment(p, large_set)

Done importing
Start generating shifts, shiftsize is 109
Created independent set of size 325 with q_array [      1      39    1521   59319 2313441]
Found 80 available points to extend the independent set.
Graph has 80 nodes and 141 edges


KeyboardInterrupt: 

518400


## Search for 3 permutations with distances in {4,5,6}

We fix the first permutation as:

\[
P_1 = (1,2,3,4,5,6,7,8,9,10)
\]

and require the other two permutations to also start with 1.

For each permutation, exactly 15 unordered number-pairs are at position distance 4, 5, or 6.
There are \(\binom{10}{2}=45\) unordered pairs total, so if 3 permutations satisfy the condition for every pair, the 3 sets of 15 covered pairs must partition all 45 pairs.

The code below:
1. Enumerates all permutations of 2..10 appended after 1.
2. Computes the 15-pair coverage mask for each permutation.
3. Searches for \(P_2, P_3\) so that masks of \(P_1, P_2, P_3\) are disjoint and their union is all pairs.

In [5]:
from itertools import permutations

n = 10
D = {4, 5, 6}
values = tuple(range(1, n + 1))

# Index unordered pairs (a,b) with a<b into bit positions 0..44.
pair_to_bit = {}
all_pairs = []
bit = 0
for a in range(1, n + 1):
    for b in range(a + 1, n + 1):
        pair_to_bit[(a, b)] = bit
        all_pairs.append((a, b))
        bit += 1

FULL_MASK = (1 << len(all_pairs)) - 1


def coverage_mask(perm):
    """Return a bitmask of pairs at distance 4,5,6 in this permutation."""
    pos = {v: i for i, v in enumerate(perm)}
    mask = 0
    for a, b in all_pairs:
        if abs(pos[a] - pos[b]) in D:
            mask |= 1 << pair_to_bit[(a, b)]
    return mask


P1 = tuple(range(1, 11))
M1 = coverage_mask(P1)

# Enumerate all permutations that start with 1 and cache one witness per mask.
mask_to_perm = {}
all_entries = []
for tail in permutations(range(2, 11)):
    p = (1,) + tail
    m = coverage_mask(p)
    all_entries.append((m, p))
    if m not in mask_to_perm:
        mask_to_perm[m] = p

solution = None
for m2, p2 in all_entries:
    # If M1 and M2 overlap, then we cannot partition all 45 pairs into 3 blocks of 15.
    if M1 & m2:
        continue

    target_m3 = FULL_MASK ^ M1 ^ m2

    # Need an actual permutation with exactly this target mask.
    p3 = mask_to_perm.get(target_m3)
    if p3 is None:
        continue

    # Sanity check (should be true by construction if target mask exists).
    m3 = coverage_mask(p3)
    if (M1 | m2 | m3) == FULL_MASK and (M1 & m2) == 0 and (M1 & m3) == 0 and (m2 & m3) == 0:
        solution = (P1, p2, p3)
        break

if solution is None:
    print("No such 3 permutations exist under these constraints.")
else:
    P1_sol, P2_sol, P3_sol = solution
    print("Found a valid triple:")
    print("P1 =", P1_sol)
    print("P2 =", P2_sol)
    print("P3 =", P3_sol)

    # Extra verification in the original 'at least one permutation' form.
    pos_list = [{v: i for i, v in enumerate(P)} for P in (P1_sol, P2_sol, P3_sol)]
    ok = True
    bad_pair = None
    for a, b in all_pairs:
        if not any(abs(pos[a] - pos[b]) in D for pos in pos_list):
            ok = False
            bad_pair = (a, b)
            break

    print("Verification:", "PASSED" if ok else f"FAILED at pair {bad_pair}")

Found a valid triple:
P1 = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
P2 = (1, 2, 5, 6, 7, 4, 3, 8, 9, 10)
P3 = (1, 3, 6, 2, 4, 7, 8, 9, 5, 10)
Verification: PASSED
